# T3.3 EIT-001 Environmental Monitoring Tool

READ.ME

This notebook calls the databases and project defined in EIT-003 and handles the import and integration of sensor data into the LCIs.

- Setting the project 
- Relevant sensor data manipulation 
- Integration in foreground database as scenario
- LCA results in DPP format

> The functionalities represent the requirements for EU REED WP3 T3.3 EIT-001 deliverable. Notebook depends on 1) EIT-003 Environmental Performance Model

### 1. Set the project

In [108]:
import bw2data as bd
import bw2calc as bc
import bw2io as bi
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import bw2analyzer as bwa
from reed.utils import Config
import bw_processing as bwp
from bw2io.strategies.generic import set_code_by_activity_hash


import matplotlib.pyplot as plt
import d3graph
import polyviz

import json
from datetime import date

In [109]:
config = Config()

In [110]:
bd.projects.set_current(config.PROJECT_NAME)

In [111]:
bonsai = bd.Database('BONSAI_V2.1.6')
bonsai_bio = bd.Database('BONSAI_V2.1.6 biosphere')
ecoinvent = bd.Database('ecoinvent-3.10-biosphere')
reed = bd.Database('reed')

### 2. Sensor data integration

#### 2.1 Import sensor data

This part draws from defined functions in the script, sensordata_analysis.py. It imports the monitoring data and calculates the energy consumption per tool and connect the tools to the reed tools

In [112]:
from reed.sensordata_analysis import (
    run_sensordata_analysis,
    find_missing_op_type_tools,
    compute_event_totals,
    split_final_df_by_company_event,
    sum_by_reed_tool,
    split_final_df_by_company_event,
)

In [113]:
monitoring_data_path = config.REED_PATH/'Data'/"MonitoringData"/"Data"
tool_descriptions_path = config.REED_PATH/'Data'/"MonitoringData"

In [114]:
sensordata, event_summaries_df, ambiguous_company_tools, key_company_tools, key_reed_tools = run_sensordata_analysis(
    monitoring_data_path,
    tool_descriptions_path,
)
print(sensordata)

C:\Users\TimWeber\repos_20LCA\REED\reed\src\reed\sensordata_analysis.py:144: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)


                                                  energy_consumption  \
company event_id                     Tool_Number                       
goimek  OF10001_G_BQC_S8CF2G         0.0                    8.154403   
                                     1.0                   24.300319   
                                     2.0                   10.289347   
                                     3.0                   14.673875   
                                     4.0                    0.930889   
...                                                              ...   
wwr     OF10005_SETUPS2_G_TWB_7SZ421 2694.0                 2.915730   
                                     2695.0                 1.068689   
                                     2696.0                 2.709187   
                                     2722.0                 0.631658   
                                     3350.0                23.502143   

                                                  Power_Active 

Identify tools that have different operation type in the key_company_tools. These tool numbers are removed from the key to avoid an ambigious key. These tool numbers will be assigned to the generic machining process in the reed database

In [115]:
print(ambiguous_company_tools)

   company  Tool_Number               op_type_candidates
0   goimek           31               [CHAMFER, MILLING]
1  komatsu           50    [ROUGH BORING, ROUGH MILLING]
2  komatsu           72  [FINISH MILLING, ROUGH MILLING]


In [116]:
#Identify missing op type tools, e.g. tools that are in the monitoring data but not in tool descriptions
missing_tool_descriptions = find_missing_op_type_tools(sensordata)

In [117]:
#Compute total time and energy per event per company
event_totals = compute_event_totals(sensordata, event_summaries_df)

#### 2.2 Scenario setup for sensor data integation

this section is missing a part about how to handle the processes that are not in hours?

In [118]:
from reed.swapping_data import (
    find_activities_by_name,
    build_scenarios_from_power_active,
    build_consumer_input_scenarios_from_hours,
    describe_scenarios,
    lcia_for_activity_with_scenario,
    find_subactivity_by_name_fragment,
    build_consumer_input_scenarios_from_hours_for_activity,
    get_product_id_for_activity,
)

Select which event you want to analyse

In [125]:
company = "komatsu"
event_id = "OF10006_G_TXG_REGRHM"
product = "KOMATSU total"
demand = 1
product_reed = product

Start by selected the company and event_id where you want to swap in the sensordata

In [126]:
#Split sensordata by company and event
event_dfs = split_final_df_by_company_event(sensordata)

#Compute energy by reed tool for a specific company and event
tool_energy_by_company_and_event = sum_by_reed_tool(
    event_dfs[(company, event_id)]
)
print(tool_energy_by_company_and_event)

                                                          energy_consumption  \
company event_id             reed tool                                         
komatsu OF10006_G_TXG_REGRHM boring, soraluce                      19.791830   
                             generic machining, soraluce           70.825454   
                             milling, soraluce                     57.662178   

                                                              hours  \
company event_id             reed tool                                
komatsu OF10006_G_TXG_REGRHM boring, soraluce              3.441944   
                             generic machining, soraluce  16.780833   
                             milling, soraluce            10.194444   

                                                          Power_Active  
company event_id             reed tool                                  
komatsu OF10006_G_TXG_REGRHM boring, soraluce                 5.750189  
                        


Find all processes present in the sensordata in the reed-database

In [127]:
tools = tool_energy_by_company_and_event.index.get_level_values("reed tool").unique()
found = {t: find_activities_by_name(reed, t, exact=False) for t in tools}
found

{'boring, soraluce': ['boring, soraluce' (hour, DE, None),
  'boring, soraluce' (hour, GB, None),
  'boring, soraluce' (hour, ES, None)],
 'generic machining, soraluce': ['generic machining, soraluce' (kilogram, GB, None),
  'generic machining, soraluce' (hour, DE, None),
  'generic machining, soraluce' (kilogram, DE, None),
  'generic machining, soraluce' (hour, ES, None),
  'generic machining, soraluce' (hour, GB, None),
  'generic machining, soraluce' (kilogram, ES, None)],
 'milling, soraluce': ['milling, soraluce' (hour, ES, None),
  'milling, soraluce' (kilogram, ES, None),
  'milling, soraluce' (hour, DE, None),
  'milling, soraluce' (hour, GB, None),
  'milling, soraluce' (kilogram, DE, None),
  'milling, soraluce' (kilogram, GB, None)]}

Find subactivities called A3 for the product under study

In [128]:
product_activity = reed.get(name=product)

# Find the A3 activity inside KOMATSU total
a3_matches = find_subactivity_by_name_fragment(
    parent_activity=product_activity,
    name_fragment="A3",
)

print(a3_matches)
a3_activity = a3_matches[0]

['KOMATSU A3: machining' (unit, GB, None)]


In [129]:
soraluce_matches = find_subactivity_by_name_fragment(
    parent_activity=a3_activity,
    name_fragment="soraluce machining",
)

print(soraluce_matches)

soraluce_activity = soraluce_matches[0]

['Soraluce machining, Komatsu piece' (unit, GB, None)]


In [130]:
#Identifying the electricity activity in bonsai to be able to compute the original electricity input amount for the activities found above
electricity = bonsai.get(name='production of electricity (market for)', location='GB', unit='TJ')

Create scenario where alle electricity inputs from the sensordata to the processes identified in found.

In [131]:
scenarios = build_scenarios_from_power_active(tool_energy_by_company_and_event, found, electricity)
scenarios


original  komatsu_OF10006_G_TXG_REGRHM
product            activity                                                  
318672587033227265 323362257478430720       0.0                      0.000021
                   323362257386156032       0.0                      0.000015
                   323362257444876288       0.0                      0.000020

The input of products in found to consuming activities are found and used to build scenarios.

In [132]:
consumer_scenarios = build_consumer_input_scenarios_from_hours_for_activity(
    tool_energy_by_company_and_event,
    found,
    consumer_activity=soraluce_activity,
)
consumer_scenarios

,,original,komatsu_OF10006_G_TXG_REGRHM
product,activity,,
323362257470042112,323362257658785792,3.198889,3.441944
323362257369378816,323362257658785792,26.722500,16.780833
323362257419710464,323362257658785792,8.063333,10.194444


In [133]:
full_scenario = pd.concat([scenarios, consumer_scenarios])

#### 2.3 Scenario data scenario setup

In [134]:
full_scenario_readable = describe_scenarios(full_scenario)
full_scenario_readable

,product_name,product_unit,activity_name,activity_unit,original,komatsu_OF10006_G_TXG_REGRHM
0,production of electricity (market for),TJ,"boring, soraluce",hour,0.000000,0.000021
1,production of electricity (market for),TJ,"generic machining, soraluce",hour,0.000000,0.000015
2,production of electricity (market for),TJ,"milling, soraluce",hour,0.000000,0.000020
3,"boring, soraluce",hour,"Soraluce machining, Komatsu piece",unit,3.198889,3.441944
4,"generic machining, soraluce",hour,"Soraluce machining, Komatsu piece",unit,26.722500,16.780833
5,"milling, soraluce",hour,"Soraluce machining, Komatsu piece",unit,8.063333,10.194444


#### 2.4 Initial LCA scores for Monitoring Footprint

LCA score with sensor data

In [135]:
score, lca = lcia_for_activity_with_scenario(
    activity=product_activity,
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    scenario_col="original",     # "original" or None (for defined scenario)
    amount=demand
)
score

11705.528974446623

In [136]:
score, lca = lcia_for_activity_with_scenario(
    activity=product_activity,
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    scenario_col=None,     # "original" or None (for defined scenario)
    amount=demand
)
score

11589.10320966999

LCA score without sensor data

### 3. Footprint Monitoring Output (Finished=0)

During the production stage, indicated by the column "finished=0", the client wants to track total climate impacts and impacts per machining operation. The input data is expected to be aggregated per operation.

In [137]:
import importlib
import reed.LCA_functions

importlib.reload(reed.LCA_functions)
from reed.LCA_functions import (get_machining_activity, monitoring_footprint_output)

In [138]:
monitoring_footprint_output_unfinished = monitoring_footprint_output(product_activity, lcia_method=config.lcia_gwp100, company=company, event_id=event_id)
print(json.dumps(monitoring_footprint_output_unfinished, indent=2))

Exported to monitoring_komatsu_OF10006_G_TXG_REGRHM_2026-06-17.json
{
  "product": "KOMATSU total",
  "total_climate_impact_kg_co2eq": 11705.529,
  "machining_contributions": [
    {
      "operation": "milling, soraluce",
      "score": 124.463342
    },
    {
      "operation": "boring, soraluce",
      "score": 54.43149
    },
    {
      "operation": "generic machining, soraluce",
      "score": 412.480999
    },
    {
      "operation": "soraluce machine, Komatsu",
      "score": 11.584068
    }
  ]
}


In [139]:
# Export to JSON file
filename = f"Monitoring_Footprint_{company}_{event_id}_{date.today().isoformat()}.json"

with open(filename, "w") as f:
    json.dump(monitoring_footprint_output_unfinished, f, indent=2)

print(f"Exported to {filename}")

Exported to Monitoring_Footprint_komatsu_OF10006_G_TXG_REGRHM_2026-06-17.json


### 4. Final Footprint Monitoring Output: DPP Export (Finished=1)

#### 4.1 Functions for DPP alignment

In [140]:
def lcia_score_for_named_input(
    parent_activity,
    name_fragment,
    method,
    scenario_df=None,
    scenario_col=None,
    amount=1.0,
):
    """
    Calculate LCIA score for direct inputs of parent_activity whose names contain name_fragment.
    Applies scenario_df if provided.
    """
    matches = [
        exc for exc in parent_activity.technosphere()
        if name_fragment.lower() in exc.input.get("name", "").lower()
    ]

    if not matches:
        raise ValueError(
            f"No input containing '{name_fragment}' found in {parent_activity.get('name')}"
        )

    total_score = 0

    for exc in matches:
        input_activity = exc.input
        input_amount = float(exc["amount"]) * amount

        if scenario_df is None:
            fu, data_objs, _ = bd.prepare_lca_inputs(
                {input_activity: input_amount},
                method=method,
            )

            lca = bc.LCA(fu, method=method, data_objs=data_objs)
            lca.lci()
            lca.lcia()

            total_score += lca.score

        else:
            score, _ = lcia_for_activity_with_scenario(
                activity=input_activity,
                method=method,
                scenario_df=scenario_df,
                scenario_col=scenario_col,
                amount=input_amount,
            )

            total_score += score

    return total_score

In [141]:
score_A1 = lcia_score_for_named_input(
    parent_activity=product_activity,
    name_fragment="A1",
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    amount=demand,
)

score_A2 = lcia_score_for_named_input(
    parent_activity=product_activity,
    name_fragment="A2",
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    amount=demand,
)

score_A3 = lcia_score_for_named_input(
    parent_activity=product_activity,
    name_fragment="A3",
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    amount=demand,
)

In [142]:
score_total = score_A1 + score_A2 + score_A3
score_total

11589.103209669995

#### 4.2 To be shared with TUDa

In [143]:
activity = bd.Database('reed').get(name=product)
location = activity.get("location")
unit = activity.get("unit")

def make_pcf_entry(score, life_cycle_phase):
    return {
        "PcfCalculationMethods": [
            {
                "PcfCalculationMethod": "ISO 14044"
            }
        ],
        "PCF in kg CO2eq": round(score, 1),
        "ProcessName": f"{product} {life_cycle_phase}",
        "ReferenceValueForCalculation": unit,
        "QuantityOfMeasureForCalculation": demand,
        "LifeCyclePhases": [
            {
                "LifeCyclePhase": life_cycle_phase
            }
        ],
        "ExplanatoryStatement": (
            "Product Carbon Footprint associated with the cradle-to-gate life cycle phases (A1-A3) of the product, calculated using the specific production data provided by the company for this product."
        ),
        "GoodsHandoverAddress": {
            "Country": location
        },
        "PublicationDate": date.today().isoformat(),
        # "ExpirationDate": "2027-03-06"
    }

dpp_output = {
    "DPP": [
        {
            "company": company,
            "event_id": event_id,
            "product": product,
            "carbonFootprint": {
                "ProductCarbonFootprints": [
                    make_pcf_entry(score_A1, "A1"),
                    make_pcf_entry(score_A2, "A2"),
                    make_pcf_entry(score_A3, "A3"),
                    make_pcf_entry(score_A1 + score_A2 + score_A3, "A1-A3")  # optional total
                ]
            }
        }
    ]
}

dpp_output


{'DPP': [{'company': 'komatsu',
   'event_id': 'OF10006_G_TXG_REGRHM',
   'product': 'KOMATSU total',
   'carbonFootprint': {'ProductCarbonFootprints': [{'PcfCalculationMethods': [{'PcfCalculationMethod': 'ISO 14044'}],
      'PCF in kg CO2eq': 10180.6,
      'ProcessName': 'KOMATSU total A1',
      'ReferenceValueForCalculation': 'unit',
      'QuantityOfMeasureForCalculation': 1,
      'LifeCyclePhases': [{'LifeCyclePhase': 'A1'}],
      'ExplanatoryStatement': 'Product Carbon Footprint associated with the cradle-to-gate life cycle phases (A1-A3) of the product, calculated using the specific production data provided by the company for this product.',
      'GoodsHandoverAddress': {'Country': 'GB'},
      'PublicationDate': '2026-06-17'},
     {'PcfCalculationMethods': [{'PcfCalculationMethod': 'ISO 14044'}],
      'PCF in kg CO2eq': 200.4,
      'ProcessName': 'KOMATSU total A2',
      'ReferenceValueForCalculation': 'unit',
      'QuantityOfMeasureForCalculation': 1,
      'LifeCycl

In [144]:
# Export to JSON file
filename = f"DPP_{company}_{event_id}_{date.today().isoformat()}.json"

with open(filename, "w") as f:
    json.dump(dpp_output, f, indent=2)

print(f"Exported to {filename}")

Exported to DPP_komatsu_OF10006_G_TXG_REGRHM_2026-06-17.json


In [145]:
bwa.print_recursive_calculation(product_activity,
lcia_method=config.lcia_gwp100,max_level=3,cutoff=0.005)

Fraction of score | Absolute score | Amount | Activity
0001 | 1.171e+04 |     1 | 'KOMATSU total' (unit, GB, None)
  0.87 | 1.018e+04 |     1 | 'KOMATSU A1: raw materials' (unit, GB, None)
    0.899 | 1.053e+04 | 2.911 | 'basic iron and steel and of ferro-alloys and first products thereof(market for)' (ton, GB, None)
      -0.0173 | -202.8 | -3.466e-05 | 'a_post|a_tair|a_taux|a_tland_all|a_twatr_all' (Meuro, WA, None)
      -0.00571 | -66.87 | -0.02191 | 'production of basic iron and steel and of ferro-alloys and first products thereof' (ton, AT,
      -0.0314 |  -368 | -0.1418 | 'production of basic iron and steel and of ferro-alloys and first products thereof' (ton, BE, No
      -0.00593 | -69.43 | -0.0225 | 'production of basic iron and steel and of ferro-alloys and first products thereof' (ton, BR, 
      -0.0491 | -574.3 | -0.1947 | 'production of basic iron and steel and of ferro-alloys and first products thereof' (ton, CN, N
      -0.00994 | -116.3 | -0.03078 | 'production of ba

In [157]:
wwr_total = bd.Database('reed').get(name = 'WWR total', location = "DE", unit = 'unit')
wwr_machining = bd.Database('reed').get(name = 'Soraluce machining, WWR', location = "DE", unit = 'unit')

In [160]:
bwa.print_recursive_calculation(wwr_machining,
lcia_method=config.lcia_gwp100,max_level=1,cutoff=0)

Fraction of score | Absolute score | Amount | Activity
0001 | 1,915 |     1 | 'Soraluce machining, WWR' (unit, DE, None)
  0.505 | 967.2 | 65.59 | 'generic machining, soraluce' (hour, DE, None)
  0.0447 | 85.67 |  3.19 | 'boring, soraluce' (hour, DE, None)
  0.0396 | 75.78 | 4.411 | 'drilling, soraluce' (hour, DE, None)
  0.399 | 763.4 | 51.77 | 'milling, soraluce' (hour, DE, None)
  0.0118 |  22.5 | 9.132e-05 | 'soraluce machine, WWR' (unit, DE, None)


In [158]:
bwa.print_recursive_calculation(wwr_total,
lcia_method=config.lcia_gwp100,max_level=1,cutoff=0)

Fraction of score | Absolute score | Amount | Activity
0001 | 4.099e+04 |     1 | 'WWR total' (unit, DE, None)
  0.93 | 3.813e+04 |     1 | 'WWR A1: material inputs' (unit, DE, None)
  0.0127 | 518.9 |     1 | 'WWR A2: transport' (unit, DE, None)
  0.0571 | 2,341 |     1 | 'WWR A3: machining' (unit, DE, None)


In [159]:
wwr_total_ca = bwa.utils.recursive_calculation_to_object(
        wwr_total,
        lcia_method=config.lcia_gwp100,
        max_level=1,
        cutoff=0,
        as_dataframe=True,
    )

wwr_total_ca

,label,parent,score,fraction,amount,name,key
0,root,None,40993.597232,1.000000,1.0,WWR total,"(reed, 4d55ff8c917c3c944685d169d1887bf2)"
1,root_a,root,38133.359143,0.930227,1.0,WWR A1: material inputs,"(reed, 4428f445499f433efd9573c4b43384dc)"
2,root_b,root,518.927112,0.012659,1.0,WWR A2: transport,"(reed, 653b3310b6d7ad00ed2cfd5a31355fa5)"
3,root_c,root,2341.310978,0.057114,1.0,WWR A3: machining,"(reed, 2685709e3b8df06c752a31e091dd31d3)"
